In [1]:
# Librerías para manipulación de datos y scraping
import pandas as pd

# Librerías para manejar el navegador y scraping de Google Maps
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Librerías para manejar esperas en Selenium
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("¡Librerías importadas correctamente!")

¡Librerías importadas correctamente!


In [2]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("¡Librerías listas!")

¡Librerías listas!


In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Configuración de Chrome
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-notifications")

# Inicializamos el driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

print("Driver de Chrome listo")

Driver de Chrome listo


In [5]:
# URL de búsqueda en Google Maps
url = "https://www.google.com/maps/search/fisioterapia+medellin"

# Abrimos la página
driver.get(url)

print("Google Maps abierto con la búsqueda")

Google Maps abierto con la búsqueda


In [6]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

# Espera hasta que aparezcan los resultados
wait = WebDriverWait(driver, 10)
results = wait.until(EC.presence_of_all_elements_located(
    (By.XPATH, '//div[@role="article"]')
))

# Lista para guardar datos
leads = []

# Extraemos nombre y dirección de los primeros 10 resultados como ejemplo
for result in results[:10]:
    try:
        name = result.find_element(By.XPATH, './/h3').text
    except:
        name = ""
    try:
        address = result.find_element(By.XPATH, './/span[contains(@class,"section-result-location")]').text
    except:
        address = ""
    leads.append({"Nombre": name, "Dirección": address})

# Guardamos en un DataFrame
df = pd.DataFrame(leads)
df

,Nombre,Dirección
0,,
1,,
2,,
3,,
4,,
5,,
6,,


In [8]:
import time

# Espera unos segundos para que la página cargue completamente
time.sleep(5)

# Buscamos los resultados visibles
results = driver.find_elements(By.XPATH, '//div[contains(@aria-label,"Resultados de búsqueda")]//div[@role="article"]')

leads = []

for result in results[:10]:  # primeros 10 resultados
    try:
        name = result.find_element(By.XPATH, './/div[@role="heading"]').text
    except:
        name = ""
    try:
        address = result.find_element(By.XPATH, './/span[contains(text(),"Medellín")]').text
    except:
        address = ""
    leads.append({"Nombre": name, "Dirección": address})

# Guardamos en DataFrame
df = pd.DataFrame(leads)
df

""


In [9]:
import googlemaps
import pandas as pd
import time

# Tu API Key
API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"

# Conexión con la API
gmaps = googlemaps.Client(key=API_KEY)

# Lista para guardar todos los leads
leads = []

# Primer búsqueda: centros odontológicos en Medellín
query = "centros odontológicos Medellín"
results = gmaps.places(query=query, type="dentist", language="es")

# Función para extraer detalles de cada lugar
def get_place_details(place_id):
    details = gmaps.place(place_id=place_id, fields=['formatted_phone_number','website'])
    phone = details.get('result', {}).get('formatted_phone_number', '')
    website = details.get('result', {}).get('website', '')
    return phone, website

# Extraer datos de la primera página de resultados
for place in results['results']:
    name = place.get('name', '')
    address = place.get('formatted_address', '')
    rating = place.get('rating', '')
    place_id = place.get('place_id', '')
    
    phone, website = get_place_details(place_id)
    
    leads.append({
        "Nombre": name,
        "Dirección": address,
        "Teléfono": phone,
        "Web": website,
        "Rating": rating
    })

# Guardar en Excel
df = pd.DataFrame(leads)
df.to_excel("leads_odontologos_medellin.xlsx", index=False)

df

ModuleNotFoundError: No module named 'googlemaps'

In [11]:
!pip install googlemaps pandas --upgrade

In [12]:
import sys
print(sys.executable)

C:\Users\LENOVO\AppData\Local\Programs\Python\Python314\python.exe


In [13]:
import googlemaps
import pandas as pd
import time

print("¡Librerías cargadas correctamente!")

¡Librerías cargadas correctamente!


In [15]:
import requests
import pandas as pd
import time

# -----------------------------
# 1️⃣ Tu API Key (Places API Nueva)
# -----------------------------
API_KEY = "TU_API_KEY_AQUI"

# -----------------------------
# 2️⃣ Parámetros de búsqueda
# -----------------------------
query = "centros odontológicos Medellín"
url_search = "https://maps.googleapis.com/maps/api/place/textsearch/json"

leads = []

# -----------------------------
# 3️⃣ Paginación automática
# -----------------------------
params = {
    "query": query,
    "key": API_KEY,
    "language": "es"
}

while True:
    response = requests.get(url_search, params=params)
    data = response.json()
    
    for place in data.get("results", []):
        name = place.get("name", "")
        address = place.get("formatted_address", "")
        rating = place.get("rating", "")
        place_id = place.get("place_id", "")
        
        # Detalles adicionales: teléfono y web
        url_details = "https://maps.googleapis.com/maps/api/place/details/json"
        details_params = {
            "place_id": place_id,
            "fields": "formatted_phone_number,website",
            "key": API_KEY,
            "language": "es"
        }
        details_resp = requests.get(url_details, params=details_params).json()
        phone = details_resp.get("result", {}).get("formatted_phone_number", "")
        website = details_resp.get("result", {}).get("website", "")
        
        leads.append({
            "Nombre": name,
            "Dirección": address,
            "Teléfono": phone,
            "Web": website,
            "Rating": rating
        })
    
    # Revisar si hay más páginas
    next_page_token = data.get("next_page_token")
    if next_page_token:
        time.sleep(2)  # Espera para que Google genere la siguiente página
        params = {"pagetoken": next_page_token, "key": API_KEY, "language": "es"}
    else:
        break

# -----------------------------
# 4️⃣ Guardar en Excel
# -----------------------------
df = pd.DataFrame(leads)
df.to_excel("leads_odontologos_medellin.xlsx", index=False)
df

ModuleNotFoundError: No module named 'openpyxl'

In [16]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

In [18]:
df.to_excel("leads_odontologos_medellin.xlsx", index=False)

In [19]:
import requests
import pandas as pd

API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
query = "centros odontológicos Medellín"
url_search = "https://maps.googleapis.com/maps/api/place/textsearch/json"

params = {
    "query": query,
    "key": API_KEY,
    "language": "es"
}

response = requests.get(url_search, params=params).json()
print("Status:", response.get("status"))
print("Resultados encontrados:", len(response.get("results", [])))

Status: REQUEST_DENIED
Resultados encontrados: 0


In [20]:
import requests
import pandas as pd

API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
query = "centros odontológicos Medellín"
url_search = "https://maps.googleapis.com/maps/api/place/textsearch/json"

params = {
    "query": query,
    "key": API_KEY,
    "language": "es"
}

response = requests.get(url_search, params=params).json()
print("Status:", response.get("status"))
print("Resultados encontrados:", len(response.get("results", [])))
```

Debe salir:
```
Status: OK
Resultados encontrados: 20

SyntaxError: invalid syntax (562237009.py, line 17)

In [21]:
import requests
import pandas as pd

API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
query = "centros odontológicos Medellín"
url_search = "https://maps.googleapis.com/maps/api/place/textsearch/json"

params = {
    "query": query,
    "key": API_KEY,
    "language": "es"
}

response = requests.get(url_search, params=params).json()
print("Status:", response.get("status"))
print("Resultados encontrados:", len(response.get("results", [])))

Status: OK
Resultados encontrados: 20


In [22]:
import requests
import pandas as pd
import time

API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
query = "centros odontológicos Medellín"
url_search = "https://maps.googleapis.com/maps/api/place/textsearch/json"
url_details = "https://maps.googleapis.com/maps/api/place/details/json"

leads = []
params = {"query": query, "key": API_KEY, "language": "es"}

while True:
    response = requests.get(url_search, params=params).json()
    
    for place in response.get("results", []):
        place_id = place.get("place_id", "")
        
        details = requests.get(url_details, params={
            "place_id": place_id,
            "fields": "formatted_phone_number,website",
            "key": API_KEY
        }).json()
        
        leads.append({
            "Nombre": place.get("name", ""),
            "Dirección": place.get("formatted_address", ""),
            "Teléfono": details.get("result", {}).get("formatted_phone_number", ""),
            "Web": details.get("result", {}).get("website", ""),
            "Rating": place.get("rating", "")
        })
        print(f"✅ {place.get('name')}")
    
    next_page_token = response.get("next_page_token")
    if next_page_token:
        time.sleep(3)
        params = {"pagetoken": next_page_token, "key": API_KEY}
    else:
        break

df = pd.DataFrame(leads)
df.to_excel("leads_odontologos_medellin.xlsx", index=False)
print(f"\n✅ Total leads guardados: {len(leads)}")
df

✅ Centro odontológico KMO Smile Especialistas
✅ Premium Dental Medellín
✅ Clínica Dental Home
✅ ORAL CONCEPT INVISALIGN
✅ Odontología la 78 Urgencia las 24 horas
✅ Odontoespecialistas
✅ Dra. Ana Zuluaga - Diseño de Sonrisa en Medellín
✅ Dental Center
✅ Consultorio odontologico la 73
✅ Infinity Smile
✅ Oral Studio
✅ Dentioral - Sede Poblado
✅ Sanadent Odontólogos - La Mejor Clínica Dental en Medellín
✅ CR Dental Studio
✅ Dental Expertos
✅ Royal Dental Care
✅ Bioestética Dental
✅ ORAL MAS ESPECIALISTAS
✅ Odontología 30a
✅ Clínica Viena / Veneers Medellín
✅ Ortounion Clínica Odontológica
✅ Consultorio Odontológico Vida Oral
✅ Centro Odontológico San Martin
✅ URGENCIAS ODONTOLOGICAS, MEDELLÍN
✅ Smile Natural Studio | Clínica & Spa Odontológico
✅ Centro Radiológico Oral Centro de Medellin
✅ DentiOral - Sede Belén
✅ Odontología The Best Smile Centro Medellín
✅ Dra Liliana Gutiérrez L.
✅ Bocas&Risas - Clínica Odontológica en Medellín
✅ Odontoss Centro
✅ Oral Center Poblado
✅ Oral Elite Consul

,Nombre,Dirección,Teléfono,Web,Rating
0,Centro odontológico KMO Smile Especialistas,"Av. El Poblado #cl 7 39-290, El Poblado, Medel...",314 6628875,,4.9
1,Premium Dental Medellín,"Local 2101. CC Premium Plaza, El Poblado, Mede...",320 4010740,http://premiumdentalmedellin.com/,4.9
2,Clínica Dental Home,"Cl. 34 #34A-76 Consultorio 401, Laureles - Est...",310 4194654,https://clinicadentalhome.com/?utm_source=goog...,4.9
3,ORAL CONCEPT INVISALIGN,"Cra. 70 #32 b 51 local 1, Medellín, Belén, Med...",313 7662008,https://oralconcept.co/home/oral-concept,5.0
4,Odontología la 78 Urgencia las 24 horas,"Cl. 45A #77 a 10, Los Pinos, Medellín, Laurele...",312 2478447,https://consultorioodontologicola78.com/,4.3
5,Odontoespecialistas,"Calle 34 ##43-66 Local 1008 1009, Medellín, An...",313 7591236,http://www.odontoespecialistas.com/,5.0
6,Dra. Ana Zuluaga - Diseño de Sonrisa en Medellín,"Cra. 43A #18sur-135 piso 9, oficina 910, El Po...",305 4566979,http://www.dranazuluaga.com/,4.9
7,Dental Center,"Cl. 11 #21, El Poblado, Medellín, El Poblado, ...",(604) 2686922,http://www.dentalcenter.com.co/,4.9
8,Consultorio odontologico la 73,"AKT Colombia, Cl 44 #73 - 12 segundo piso, Lau...",318 7501226,http://www.odontologosla73.com/,4.9
9,Infinity Smile,"Cra. 78A #34-41, Laureles - Estadio, Medellín,...",323 4650026,https://infinitysmile.com.co/,4.9


In [24]:
import requests
import re
import pandas as pd

df = pd.read_excel("leads_odontologos_medellin.xlsx")

def extraer_email(url):
    if not url or pd.isna(url):
        return ""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=8)
        emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', response.text)
        emails = [e for e in emails if not any(x in e for x in ['sentry', 'wix', 'google', 'schema', 'example'])]
        return emails[0] if emails else ""
    except:
        return ""

print("Extrayendo emails...")
df["Email"] = ""

for i, row in df.iterrows():
    email = extraer_email(row.get("Web"))
    df.at[i, "Email"] = email
    estado = "OK" if email else "sin email"
    print(f"{row['Nombre']} --> {email if email else estado}")

df.to_excel("leads_odontologos_medellin.xlsx", index=False)
print(f"\nEmails encontrados: {df['Email'].ne('').sum()} de {len(df)}")
df

Extrayendo emails...
Centro odontológico KMO Smile Especialistas --> sin email
Premium Dental Medellín --> premiumdentalmedellin@gmail.com
Clínica Dental Home --> info@clinicadentalhome.net
ORAL CONCEPT INVISALIGN --> sin email
Odontología la 78 Urgencia las 24 horas --> sin email
Odontoespecialistas --> gerencia@odontoespecialistas.com
Dra. Ana Zuluaga - Diseño de Sonrisa en Medellín --> bg-vertical-tabs@2x.png
Dental Center --> Pacientesdc@gmail.com
Consultorio odontologico la 73 --> odontologosla73@gmail.com
Infinity Smile --> sin email
Oral Studio --> correo@correo.com
Dentioral - Sede Poblado --> info@dentioral.com
Sanadent Odontólogos - La Mejor Clínica Dental en Medellín --> sin email
CR Dental Studio --> sin email
Dental Expertos --> sin email
Royal Dental Care --> Mesa-de-trabajo-12@2x.png
Bioestética Dental --> bioesteticadental69@gmail.com
ORAL MAS ESPECIALISTAS --> sin email
Odontología 30a --> sin email
Clínica Viena / Veneers Medellín --> gerencia@clinicaviena.com.co
Orto

,Nombre,Dirección,Teléfono,Web,Rating,Email
0,Centro odontológico KMO Smile Especialistas,"Av. El Poblado #cl 7 39-290, El Poblado, Medel...",314 6628875,NaN,4.9,
1,Premium Dental Medellín,"Local 2101. CC Premium Plaza, El Poblado, Mede...",320 4010740,http://premiumdentalmedellin.com/,4.9,premiumdentalmedellin@gmail.com
2,Clínica Dental Home,"Cl. 34 #34A-76 Consultorio 401, Laureles - Est...",310 4194654,https://clinicadentalhome.com/?utm_source=goog...,4.9,info@clinicadentalhome.net
3,ORAL CONCEPT INVISALIGN,"Cra. 70 #32 b 51 local 1, Medellín, Belén, Med...",313 7662008,https://oralconcept.co/home/oral-concept,5.0,
4,Odontología la 78 Urgencia las 24 horas,"Cl. 45A #77 a 10, Los Pinos, Medellín, Laurele...",312 2478447,https://consultorioodontologicola78.com/,4.3,
5,Odontoespecialistas,"Calle 34 ##43-66 Local 1008 1009, Medellín, An...",313 7591236,http://www.odontoespecialistas.com/,5.0,gerencia@odontoespecialistas.com
6,Dra. Ana Zuluaga - Diseño de Sonrisa en Medellín,"Cra. 43A #18sur-135 piso 9, oficina 910, El Po...",305 4566979,http://www.dranazuluaga.com/,4.9,bg-vertical-tabs@2x.png
7,Dental Center,"Cl. 11 #21, El Poblado, Medellín, El Poblado, ...",(604) 2686922,http://www.dentalcenter.com.co/,4.9,Pacientesdc@gmail.com
8,Consultorio odontologico la 73,"AKT Colombia, Cl 44 #73 - 12 segundo piso, Lau...",318 7501226,http://www.odontologosla73.com/,4.9,odontologosla73@gmail.com
9,Infinity Smile,"Cra. 78A #34-41, Laureles - Estadio, Medellín,...",323 4650026,https://infinitysmile.com.co/,4.9,


In [25]:
import requests
import pandas as pd
import time
import re

API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
url_search = "https://maps.googleapis.com/maps/api/place/textsearch/json"
url_details = "https://maps.googleapis.com/maps/api/place/details/json"

# Todas las zonas de busqueda
zonas = [
    "centros odontologicos El Poblado Medellin",
    "centros odontologicos Laureles Medellin",
    "centros odontologicos Envigado",
    "centros odontologicos Bello",
    "centros odontologicos Itagui",
    "centros odontologicos Centro Medellin",
    "centros odontologicos Sabaneta",
    "centros odontologicos Belen Medellin",
    "centros odontologicos Robledo Medellin",
    "centros odontologicos Castilla Medellin"
]

todos_leads = []
ids_vistos = set()

def extraer_email(url):
    if not url or pd.isna(url):
        return ""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=8)
        emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', response.text)
        emails = [e for e in emails if not any(x in e for x in ['sentry', 'wix', 'google', 'schema', 'example'])]
        return emails[0] if emails else ""
    except:
        return ""

for zona in zonas:
    print(f"\nBuscando: {zona}")
    params = {"query": zona, "key": API_KEY, "language": "es"}
    
    while True:
        response = requests.get(url_search, params=params).json()
        
        for place in response.get("results", []):
            place_id = place.get("place_id", "")
            
            if place_id in ids_vistos:
                continue
            ids_vistos.add(place_id)
            
            details = requests.get(url_details, params={
                "place_id": place_id,
                "fields": "formatted_phone_number,website",
                "key": API_KEY
            }).json()
            
            web = details.get("result", {}).get("website", "")
            email = extraer_email(web)
            
            todos_leads.append({
                "Nombre": place.get("name", ""),
                "Direccion": place.get("formatted_address", ""),
                "Telefono": details.get("result", {}).get("formatted_phone_number", ""),
                "Web": web,
                "Rating": place.get("rating", ""),
                "Email": email,
                "Zona": zona
            })
            print(f"OK: {place.get('name')}")
        
        next_page_token = response.get("next_page_token")
        if next_page_token:
            time.sleep(3)
            params = {"pagetoken": next_page_token, "key": API_KEY}
        else:
            break

df = pd.DataFrame(todos_leads)
df.to_excel("leads_odontologos_medellin_completo.xlsx", index=False)
print(f"\nTotal leads unicos guardados: {len(todos_leads)}")
df


Buscando: centros odontologicos El Poblado Medellin
OK: Oral Center Poblado
OK: Dental Center
OK: Centro odontológico KMO Smile Especialistas
OK: Poblado Dental
OK: Odontología el Poblado - Oral 10 - Diseños de Sonrisa en Medellín
OK: ORAL DESIGN POBLADO
OK: Oral Studio
OK: Dra. Maru Odontología y Estética Facial
OK: Dra. Ana Zuluaga - Diseño de Sonrisa en Medellín
OK: Smile From Medellin - Dental Clinic
OK: DentiSalud El Poblado - Clínicas Odontológicas
OK: CR Dental Studio
OK: Royal Dental Care
OK: W SMILE premium dental practice
OK: Premium Dental Medellín
OK: Smile Natural Studio | Clínica & Spa Odontológico
OK: Clínica Viena / Veneers Medellín
OK: MY DENTAL
OK: Dentioral - Sede Poblado
OK: Present Dental Studio
OK: Dra. Carolina Bermúdez
OK: Centro Vital, Odontología Integral
OK: Clínica Unilaser
OK: Valo Dental Studio
OK: MDental Láser
OK: Dora Álvarez
OK: Focus Dental Center Odontología
OK: Dental Specialists Medellín
OK: Jenny Villada Clínicas Odontológicas
OK: Smile Dental Ce

,Nombre,Direccion,Telefono,Web,Rating,Email,Zona
0,Oral Center Poblado,"Calle 7 # 39- 197, Torre Intermédica Local 121...",(604) 3493080,http://oralcenter.com.co/,4.4,,centros odontologicos El Poblado Medellin
1,Dental Center,"Cl. 11 #21, El Poblado, Medellín, El Poblado, ...",(604) 2686922,http://www.dentalcenter.com.co/,4.9,Pacientesdc@gmail.com,centros odontologicos El Poblado Medellin
2,Centro odontológico KMO Smile Especialistas,"Av. El Poblado #cl 7 39-290, El Poblado, Medel...",314 6628875,,4.9,,centros odontologicos El Poblado Medellin
3,Poblado Dental,"Cl 10 #42-45 Oficina 229, El Poblado, Medellín...",324 6027040,https://pobladodental.com/,4.6,pobladodental@gmail.com,centros odontologicos El Poblado Medellin
4,Odontología el Poblado - Oral 10 - Diseños de ...,"Cra. 48 #7 270, El Poblado, Medellín, El Pobla...",301 1795872,http://www.oral10odontologia.com/,5,oral10vegas@gmail.com,centros odontologicos El Poblado Medellin
...,...,...,...,...,...,...,...
370,MJ Odontología Integral,"Cl. 91A # 71 - 30 int 301, Francisco Zea, Mede...",305 3818475,,3,,centros odontologicos Castilla Medellin
371,Odontología Especializada,"Cl. 57 #49-44, La Candelaria, Medellín, La Can...",333 2922519,,5,,centros odontologicos Castilla Medellin
372,Odontología Aranjuez,"Cra. 49A #92-90, Aranjuez, Medellín, Aranjuez,...",310 8388449,https://wa.me/573108388449,4.9,,centros odontologicos Castilla Medellin
373,RxNortDental,"CC Terminal del Norte, Local 156, Oleoducto, M...",,,3.4,,centros odontologicos Castilla Medellin


In [26]:
import requests
import pandas as pd
import re
from urllib.parse import urljoin, urlparse

df = pd.read_excel("leads_odontologos_medellin_completo.xlsx")

def extraer_email_profundo(url):
    if not url or pd.isna(url):
        return ""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        
        # Primero buscar en la pagina principal
        response = requests.get(url, headers=headers, timeout=8)
        emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', response.text)
        emails = [e for e in emails if not any(x in e for x in ['sentry', 'wix', 'google', 'schema', 'example', 'jquery'])]
        if emails:
            return emails[0]
        
        # Si no hay email, buscar en paginas de contacto
        paginas_contacto = [
            urljoin(url, "/contacto"),
            urljoin(url, "/contact"),
            urljoin(url, "/nosotros"),
            urljoin(url, "/about"),
            urljoin(url, "/contactenos"),
        ]
        
        for pagina in paginas_contacto:
            try:
                r = requests.get(pagina, headers=headers, timeout=6)
                if r.status_code == 200:
                    emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', r.text)
                    emails = [e for e in emails if not any(x in e for x in ['sentry', 'wix', 'google', 'schema', 'example', 'jquery'])]
                    if emails:
                        return emails[0]
            except:
                continue
        return ""
    except:
        return ""

print("Mejorando extraccion de emails...")

for i, row in df.iterrows():
    if row.get("Email") and not pd.isna(row.get("Email")):
        print(f"YA TIENE: {row['Nombre']} --> {row['Email']}")
        continue
    
    email = extraer_email_profundo(row.get("Web"))
    df.at[i, "Email"] = email
    estado = email if email else "sin email"
    print(f"{row['Nombre']} --> {estado}")

df.to_excel("leads_odontologos_medellin_completo.xlsx", index=False)
print(f"\nTotal emails: {df['Email'].ne('').sum()} de {len(df)}")
df

Mejorando extraccion de emails...
Oral Center Poblado --> sin email
YA TIENE: Dental Center --> Pacientesdc@gmail.com
Centro odontológico KMO Smile Especialistas --> sin email
YA TIENE: Poblado Dental --> pobladodental@gmail.com
YA TIENE: Odontología el Poblado - Oral 10 - Diseños de Sonrisa en Medellín --> oral10vegas@gmail.com
ORAL DESIGN POBLADO --> sin email
YA TIENE: Oral Studio --> correo@correo.com
Dra. Maru Odontología y Estética Facial --> sin email
YA TIENE: Dra. Ana Zuluaga - Diseño de Sonrisa en Medellín --> bg-vertical-tabs@2x.png
Smile From Medellin - Dental Clinic --> sin email
YA TIENE: DentiSalud El Poblado - Clínicas Odontológicas --> info@dentisalud.com.co
CR Dental Studio --> sin email
YA TIENE: Royal Dental Care --> Mesa-de-trabajo-12@2x.png
W SMILE premium dental practice --> sin email
YA TIENE: Premium Dental Medellín --> premiumdentalmedellin@gmail.com
YA TIENE: Smile Natural Studio | Clínica & Spa Odontológico --> usuario@mail.com
YA TIENE: Clínica Viena / Vene

,Nombre,Direccion,Telefono,Web,Rating,Email,Zona
0,Oral Center Poblado,"Calle 7 # 39- 197, Torre Intermédica Local 121...",(604) 3493080,http://oralcenter.com.co/,4.4,,centros odontologicos El Poblado Medellin
1,Dental Center,"Cl. 11 #21, El Poblado, Medellín, El Poblado, ...",(604) 2686922,http://www.dentalcenter.com.co/,4.9,Pacientesdc@gmail.com,centros odontologicos El Poblado Medellin
2,Centro odontológico KMO Smile Especialistas,"Av. El Poblado #cl 7 39-290, El Poblado, Medel...",314 6628875,NaN,4.9,,centros odontologicos El Poblado Medellin
3,Poblado Dental,"Cl 10 #42-45 Oficina 229, El Poblado, Medellín...",324 6027040,https://pobladodental.com/,4.6,pobladodental@gmail.com,centros odontologicos El Poblado Medellin
4,Odontología el Poblado - Oral 10 - Diseños de ...,"Cra. 48 #7 270, El Poblado, Medellín, El Pobla...",301 1795872,http://www.oral10odontologia.com/,5.0,oral10vegas@gmail.com,centros odontologicos El Poblado Medellin
...,...,...,...,...,...,...,...
370,MJ Odontología Integral,"Cl. 91A # 71 - 30 int 301, Francisco Zea, Mede...",305 3818475,NaN,3.0,,centros odontologicos Castilla Medellin
371,Odontología Especializada,"Cl. 57 #49-44, La Candelaria, Medellín, La Can...",333 2922519,NaN,5.0,,centros odontologicos Castilla Medellin
372,Odontología Aranjuez,"Cra. 49A #92-90, Aranjuez, Medellín, Aranjuez,...",310 8388449,https://wa.me/573108388449,4.9,,centros odontologicos Castilla Medellin
373,RxNortDental,"CC Terminal del Norte, Local 156, Oleoducto, M...",NaN,NaN,3.4,,centros odontologicos Castilla Medellin


In [28]:
import requests
import pandas as pd
import time

API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
url_search = "https://maps.googleapis.com/maps/api/place/textsearch/json"

df_existente = pd.read_excel("leads_odontologos_medellin_completo.xlsx")

zonas = [
    "centros odontologicos El Poblado Medellin",
    "centros odontologicos Laureles Medellin",
    "centros odontologicos Envigado",
    "centros odontologicos Bello",
    "centros odontologicos Itagui",
    "centros odontologicos Centro Medellin",
    "centros odontologicos Sabaneta",
    "centros odontologicos Belen Medellin",
    "centros odontologicos Robledo Medellin",
    "centros odontologicos Castilla Medellin"
]

# Guardar place_ids existentes si ya tiene esa columna
if "place_id" in df_existente.columns:
    ids_vistos = set(df_existente["place_id"].dropna().tolist())
else:
    ids_vistos = set()

nuevos_ids = []

for zona in zonas:
    params = {"query": zona, "key": API_KEY, "language": "es"}
    while True:
        response = requests.get(url_search, params=params).json()
        for place in response.get("results", []):
            place_id = place.get("place_id", "")
            nuevos_ids.append({
                "Nombre": place.get("name", ""),
                "place_id": place_id
            })
        next_page_token = response.get("next_page_token")
        if next_page_token:
            time.sleep(3)
            params = {"pagetoken": next_page_token, "key": API_KEY}
        else:
            break

# Agregar place_id al Excel existente haciendo match por nombre
df_ids = pd.DataFrame(nuevos_ids).drop_duplicates(subset="Nombre")
df_final = df_existente.merge(df_ids, on="Nombre", how="left")
df_final.to_excel("leads_odontologos_medellin_completo.xlsx", index=False)
print(f"place_id agregado a {df_final['place_id'].notna().sum()} de {len(df_final)} registros")
df_final

KeyError: 'place_id'

In [29]:
# Ver columnas que tiene el Excel actualmente
df_existente = pd.read_excel("leads_odontologos_medellin_completo.xlsx")
print(df_existente.columns.tolist())

['Nombre', 'Direccion', 'Telefono', 'Web', 'Rating', 'Email', 'Zona', 'place_id_x', 'place_id_y']


In [30]:
import pandas as pd

df = pd.read_excel("leads_odontologos_medellin_completo.xlsx")

# Combinar las dos columnas en una sola
df["place_id"] = df["place_id_x"].fillna(df["place_id_y"])

# Eliminar las columnas duplicadas
df = df.drop(columns=["place_id_x", "place_id_y"])

df.to_excel("leads_odontologos_medellin_completo.xlsx", index=False)
print(f"place_id agregado a {df['place_id'].notna().sum()} de {len(df)} registros")
print(df.columns.tolist())
df

place_id agregado a 355 de 375 registros
['Nombre', 'Direccion', 'Telefono', 'Web', 'Rating', 'Email', 'Zona', 'place_id']


,Nombre,Direccion,Telefono,Web,Rating,Email,Zona,place_id
0,Oral Center Poblado,"Calle 7 # 39- 197, Torre Intermédica Local 121...",(604) 3493080,http://oralcenter.com.co/,4.4,NaN,centros odontologicos El Poblado Medellin,ChIJX2o-xYGCRo4Rn6L7wacsRJo
1,Dental Center,"Cl. 11 #21, El Poblado, Medellín, El Poblado, ...",(604) 2686922,http://www.dentalcenter.com.co/,4.9,Pacientesdc@gmail.com,centros odontologicos El Poblado Medellin,ChIJgwmJfiYoRI4RvO8goH3oe7M
2,Centro odontológico KMO Smile Especialistas,"Av. El Poblado #cl 7 39-290, El Poblado, Medel...",314 6628875,NaN,4.9,NaN,centros odontologicos El Poblado Medellin,ChIJc8AO2KUpRI4R5lZzeBklf3U
3,Poblado Dental,"Cl 10 #42-45 Oficina 229, El Poblado, Medellín...",324 6027040,https://pobladodental.com/,4.6,pobladodental@gmail.com,centros odontologicos El Poblado Medellin,ChIJ42xK_4kpRI4RwY-W14dE_BU
4,Odontología el Poblado - Oral 10 - Diseños de ...,"Cra. 48 #7 270, El Poblado, Medellín, El Pobla...",301 1795872,http://www.oral10odontologia.com/,5.0,oral10vegas@gmail.com,centros odontologicos El Poblado Medellin,ChIJN3fIPZ8pRI4RytlAMwH5Zuw
...,...,...,...,...,...,...,...,...
370,MJ Odontología Integral,"Cl. 91A # 71 - 30 int 301, Francisco Zea, Mede...",305 3818475,NaN,3.0,NaN,centros odontologicos Castilla Medellin,NaN
371,Odontología Especializada,"Cl. 57 #49-44, La Candelaria, Medellín, La Can...",333 2922519,NaN,5.0,NaN,centros odontologicos Castilla Medellin,ChIJZ45MIQApRI4Rkq8Ou7UWmFw
372,Odontología Aranjuez,"Cra. 49A #92-90, Aranjuez, Medellín, Aranjuez,...",310 8388449,https://wa.me/573108388449,4.9,NaN,centros odontologicos Castilla Medellin,ChIJAWOkslYpRI4RFe5kqOuwLRM
373,RxNortDental,"CC Terminal del Norte, Local 156, Oleoducto, M...",NaN,NaN,3.4,NaN,centros odontologicos Castilla Medellin,NaN


In [31]:

import requests
import pandas as pd

API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
df = pd.read_excel("leads_odontologos_medellin_completo.xlsx")

# Filtrar los que no tienen place_id
sin_id = df[df["place_id"].isna()]
print(f"Registros sin place_id: {len(sin_id)}")

for i, row in sin_id.iterrows():
    try:
        params = {
            "query": row["Nombre"] + " Medellin",
            "key": API_KEY,
            "language": "es"
        }
        response = requests.get(
            "https://maps.googleapis.com/maps/api/place/textsearch/json",
            params=params
        ).json()
        
        results = response.get("results", [])
        if results:
            df.at[i, "place_id"] = results[0].get("place_id", "")
            print(f"OK: {row['Nombre']}")
        else:
            print(f"No encontrado: {row['Nombre']}")
    except:
        print(f"Error: {row['Nombre']}")

df.to_excel("leads_odontologos_medellin_completo.xlsx", index=False)
print(f"\nplace_id total: {df['place_id'].notna().sum()} de {len(df)}")

Registros sin place_id: 20
OK: Ortoclin Sede Floresta
OK: ODONTOLOGIA EL NOGAL
OK: Space. Centro radiológico oral cero 70, la 33
OK: Sonría
OK: ORAL-TEC
OK: Vivartec - Odontología
OK: ORAL DESIGN CENTRO
OK: ORAL DESIGN - DISEÑO DE SONRISA - VENEERS
OK: Clínica Dental - Belén Dental
OK: Dental smile laboratorio
OK: Clinica Menta
OK: Dental House
OK: Dental Team - SEDE BELÉN (PLAZOLETA NUEVA VILLA DEL ABURRÁ)
OK: Ortho Estetika
OK: Mirando Sonrisas - Urgencias Odontologicas
OK: Armony Dental
OK: Consultorio Odontologico Daniela Ruiz Sanchez
OK: Laboratorio dental
OK: MJ Odontología Integral
OK: RxNortDental

place_id total: 375 de 375


In [32]:
import pandas as pd
import re

df = pd.read_excel("leads_odontologos_medellin_completo.xlsx")

def extraer_whatsapp(web, telefono):
    if web and not pd.isna(web):
        wa = re.findall(r'wa\.me/(\d+)', str(web))
        if wa:
            return "+" + wa[0]
    if telefono and not pd.isna(telefono):
        numero = re.sub(r'\D', '', str(telefono))
        if numero.startswith('3') and len(numero) == 10:
            return "+57" + numero
        if numero.startswith('573') and len(numero) == 12:
            return "+" + numero
    return ""

df["WhatsApp"] = df.apply(
    lambda row: extraer_whatsapp(row.get("Web"), row.get("Telefono")), axis=1
)

df.to_excel("leads_odontologos_medellin_completo.xlsx", index=False)
print(f"WhatsApp encontrados: {df['WhatsApp'].ne('').sum()} de {len(df)}")
df

WhatsApp encontrados: 269 de 375


,Nombre,Direccion,Telefono,Web,Rating,Email,Zona,place_id,WhatsApp
0,Oral Center Poblado,"Calle 7 # 39- 197, Torre Intermédica Local 121...",(604) 3493080,http://oralcenter.com.co/,4.4,NaN,centros odontologicos El Poblado Medellin,ChIJX2o-xYGCRo4Rn6L7wacsRJo,
1,Dental Center,"Cl. 11 #21, El Poblado, Medellín, El Poblado, ...",(604) 2686922,http://www.dentalcenter.com.co/,4.9,Pacientesdc@gmail.com,centros odontologicos El Poblado Medellin,ChIJgwmJfiYoRI4RvO8goH3oe7M,
2,Centro odontológico KMO Smile Especialistas,"Av. El Poblado #cl 7 39-290, El Poblado, Medel...",314 6628875,NaN,4.9,NaN,centros odontologicos El Poblado Medellin,ChIJc8AO2KUpRI4R5lZzeBklf3U,+573146628875
3,Poblado Dental,"Cl 10 #42-45 Oficina 229, El Poblado, Medellín...",324 6027040,https://pobladodental.com/,4.6,pobladodental@gmail.com,centros odontologicos El Poblado Medellin,ChIJ42xK_4kpRI4RwY-W14dE_BU,+573246027040
4,Odontología el Poblado - Oral 10 - Diseños de ...,"Cra. 48 #7 270, El Poblado, Medellín, El Pobla...",301 1795872,http://www.oral10odontologia.com/,5.0,oral10vegas@gmail.com,centros odontologicos El Poblado Medellin,ChIJN3fIPZ8pRI4RytlAMwH5Zuw,+573011795872
...,...,...,...,...,...,...,...,...,...
370,MJ Odontología Integral,"Cl. 91A # 71 - 30 int 301, Francisco Zea, Mede...",305 3818475,NaN,3.0,NaN,centros odontologicos Castilla Medellin,ChIJQROjti8pRI4Rqo8MLFrlSiw,+573053818475
371,Odontología Especializada,"Cl. 57 #49-44, La Candelaria, Medellín, La Can...",333 2922519,NaN,5.0,NaN,centros odontologicos Castilla Medellin,ChIJZ45MIQApRI4Rkq8Ou7UWmFw,+573332922519
372,Odontología Aranjuez,"Cra. 49A #92-90, Aranjuez, Medellín, Aranjuez,...",310 8388449,https://wa.me/573108388449,4.9,NaN,centros odontologicos Castilla Medellin,ChIJAWOkslYpRI4RFe5kqOuwLRM,+573108388449
373,RxNortDental,"CC Terminal del Norte, Local 156, Oleoducto, M...",NaN,NaN,3.4,NaN,centros odontologicos Castilla Medellin,ChIJM42n9t4pRI4RiGI30dZkmhk,


In [33]:
import requests
import pandas as pd
import time
import re

API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
url_search = "https://maps.googleapis.com/maps/api/place/textsearch/json"
url_details = "https://maps.googleapis.com/maps/api/place/details/json"

# Cargar base existente
df_existente = pd.read_excel("leads_odontologos_medellin_completo.xlsx")
ids_vistos = set(df_existente["place_id"].dropna().tolist())

# Zonas nuevas
zonas_nuevas = [
    "centros odontologicos Buenos Aires Medellin",
    "centros odontologicos Manrique Medellin",
    "centros odontologicos Aranjuez Medellin"
]

nuevos_leads = []

def extraer_email(url):
    if not url or pd.isna(url):
        return ""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=8)
        emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', response.text)
        emails = [e for e in emails if not any(x in e for x in ['sentry', 'wix', 'google', 'schema', 'example'])]
        return emails[0] if emails else ""
    except:
        return ""

def extraer_whatsapp(web, telefono):
    if web and not pd.isna(web):
        wa = re.findall(r'wa\.me/(\d+)', str(web))
        if wa:
            return "+" + wa[0]
    if telefono and not pd.isna(telefono):
        numero = re.sub(r'\D', '', str(telefono))
        if numero.startswith('3') and len(numero) == 10:
            return "+57" + numero
        if numero.startswith('573') and len(numero) == 12:
            return "+" + numero
    return ""

for zona in zonas_nuevas:
    print(f"\nBuscando: {zona}")
    params = {"query": zona, "key": API_KEY, "language": "es"}
    
    while True:
        response = requests.get(url_search, params=params).json()
        
        for place in response.get("results", []):
            place_id = place.get("place_id", "")
            
            if place_id in ids_vistos:
                print(f"Duplicado: {place.get('name')}")
                continue
            ids_vistos.add(place_id)
            
            details = requests.get(url_details, params={
                "place_id": place_id,
                "fields": "formatted_phone_number,website",
                "key": API_KEY
            }).json()
            
            web = details.get("result", {}).get("website", "")
            telefono = details.get("result", {}).get("formatted_phone_number", "")
            email = extraer_email(web)
            whatsapp = extraer_whatsapp(web, telefono)
            
            nuevos_leads.append({
                "Nombre": place.get("name", ""),
                "Direccion": place.get("formatted_address", ""),
                "Telefono": telefono,
                "Web": web,
                "Rating": place.get("rating", ""),
                "Email": email,
                "Zona": zona,
                "place_id": place_id,
                "WhatsApp": whatsapp
            })
            print(f"OK: {place.get('name')}")
        
        next_page_token = response.get("next_page_token")
        if next_page_token:
            time.sleep(3)
            params = {"pagetoken": next_page_token, "key": API_KEY}
        else:
            break

# Unir con la base existente
df_nuevos = pd.DataFrame(nuevos_leads)
df_final = pd.concat([df_existente, df_nuevos], ignore_index=True)
df_final.to_excel("leads_odontologos_medellin_completo.xlsx", index=False)
print(f"\nNuevos leads agregados: {len(nuevos_leads)}")
print(f"Total base de datos: {len(df_final)}")
df_final


Buscando: centros odontologicos Buenos Aires Medellin
Duplicado: Centro Médico Buenos Aires
Duplicado: Premium Dental Medellín
OK: ODONTOLOGÍA NOVA
Duplicado: Dental Center
OK: Centro Odontológico
OK: Ortoclin Sede Buenos Aires
Duplicado: Bocas&Risas - Clínica Odontológica en Medellín
OK: Estacion Oral
OK: Consultorio Odontologico Maria Luisa Ledezma - Urgencias Odontologicas
Duplicado: Dr. Mauricio Arias - Odontólogo Experto en Medellín English Dentist
OK: Dental By
Duplicado: Clínica Odontológica Oral Laser Centro
OK: Servicio Odontológico Cl49
Duplicado: Odontología Aranjuez
Duplicado: Focus Dental Center Odontología
OK: Odontologia Buenos Aires
Duplicado: Oral Studio
OK: International Smiles
OK: Centro Odontológico Estético
Duplicado: Oral Center Poblado
Duplicado: Odonto Rio - Estudio Dental

Buscando: centros odontologicos Manrique Medellin
OK: Dental Force Manrique
OK: Centro Médico y Odontologico Clásico
OK: Sonríes Odontología Especializada
OK: Odontología La 41
OK: Consultor

PermissionError: [Errno 13] Permission denied: 'leads_odontologos_medellin_completo.xlsx'

In [34]:
df_final.to_excel("leads_odontologos_medellin_completo.xlsx", index=False)
print(f"\nNuevos leads agregados: {len(nuevos_leads)}")
print(f"Total base de datos: {len(df_final)}")
df_final


Nuevos leads agregados: 37
Total base de datos: 412


,Nombre,Direccion,Telefono,Web,Rating,Email,Zona,place_id,WhatsApp
0,Oral Center Poblado,"Calle 7 # 39- 197, Torre Intermédica Local 121...",(604) 3493080,http://oralcenter.com.co/,4.4,NaN,centros odontologicos El Poblado Medellin,ChIJX2o-xYGCRo4Rn6L7wacsRJo,NaN
1,Dental Center,"Cl. 11 #21, El Poblado, Medellín, El Poblado, ...",(604) 2686922,http://www.dentalcenter.com.co/,4.9,Pacientesdc@gmail.com,centros odontologicos El Poblado Medellin,ChIJgwmJfiYoRI4RvO8goH3oe7M,NaN
2,Centro odontológico KMO Smile Especialistas,"Av. El Poblado #cl 7 39-290, El Poblado, Medel...",314 6628875,NaN,4.9,NaN,centros odontologicos El Poblado Medellin,ChIJc8AO2KUpRI4R5lZzeBklf3U,573146628875.0
3,Poblado Dental,"Cl 10 #42-45 Oficina 229, El Poblado, Medellín...",324 6027040,https://pobladodental.com/,4.6,pobladodental@gmail.com,centros odontologicos El Poblado Medellin,ChIJ42xK_4kpRI4RwY-W14dE_BU,573246027040.0
4,Odontología el Poblado - Oral 10 - Diseños de ...,"Cra. 48 #7 270, El Poblado, Medellín, El Pobla...",301 1795872,http://www.oral10odontologia.com/,5.0,oral10vegas@gmail.com,centros odontologicos El Poblado Medellin,ChIJN3fIPZ8pRI4RytlAMwH5Zuw,573011795872.0
...,...,...,...,...,...,...,...,...,...
407,Consultorio Odontologico El Bosque,"Cl. 80c #52-48, Aranjuez, Medellín, Aranjuez, ...",,,5,,centros odontologicos Aranjuez Medellin,ChIJYS-ib9soRI4Rmsc4Ab7rEi8,
408,Dentalov Dr Mauricio Toro,"Cra. 51 #91-17, Aranjuez, Medellín, Aranjuez, ...",318 5781173,,5,,centros odontologicos Aranjuez Medellin,ChIJ18uyOjopRI4RDfpIemTnf84,+573185781173
409,Smart Dental Group,"Cl. 73 #51D- 71 local 1013, Aranjuez, Medellín...",316 3146519,http://www.smartdentalgroup.co/,4.4,,centros odontologicos Aranjuez Medellin,ChIJKzHA4kYpRI4RTHpY-giVNH0,+573163146519
410,Consultorio Odontológico S.l.S Aranjuez,"Cra. 50 #92-21 Piso 2, Aranjuez, Medellín, Ara...",304 5246460,https://www.instagram.com/consultorioodontolog...,,,centros odontologicos Aranjuez Medellin,ChIJqXiRuUUpRI4Rg3AZahcTGYo,+573045246460


In [35]:
import requests
import pandas as pd
import time
import re

API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
url_search = "https://maps.googleapis.com/maps/api/place/textsearch/json"
url_details = "https://maps.googleapis.com/maps/api/place/details/json"

df_existente = pd.read_excel("leads_odontologos_medellin_completo.xlsx")
ids_vistos = set(df_existente["place_id"].dropna().tolist())

zonas_nuevas = [
    "centros odontologicos Copacabana Antioquia",
    "centros odontologicos La Estrella Antioquia",
    "centros odontologicos Caldas Antioquia",
    "centros odontologicos Guayabal Medellin",
    "centros odontologicos Estadio Medellin",
    "centros odontologicos Floresta Medellin",
    "centros odontologicos San Javier Medellin",
    "centros odontologicos Popular Medellin",
    "centros odontologicos Santa Cruz Medellin"
]

nuevos_leads = []

def extraer_email(url):
    if not url or pd.isna(url):
        return ""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=8)
        emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', response.text)
        emails = [e for e in emails if not any(x in e for x in ['sentry', 'wix', 'google', 'schema', 'example'])]
        return emails[0] if emails else ""
    except:
        return ""

def extraer_whatsapp(web, telefono):
    if web and not pd.isna(web):
        wa = re.findall(r'wa\.me/(\d+)', str(web))
        if wa:
            return "+" + wa[0]
    if telefono and not pd.isna(telefono):
        numero = re.sub(r'\D', '', str(telefono))
        if numero.startswith('3') and len(numero) == 10:
            return "+57" + numero
        if numero.startswith('573') and len(numero) == 12:
            return "+" + numero
    return ""

for zona in zonas_nuevas:
    print(f"\nBuscando: {zona}")
    params = {"query": zona, "key": API_KEY, "language": "es"}
    
    while True:
        response = requests.get(url_search, params=params).json()
        
        for place in response.get("results", []):
            place_id = place.get("place_id", "")
            
            if place_id in ids_vistos:
                print(f"Duplicado: {place.get('name')}")
                continue
            ids_vistos.add(place_id)
            
            details = requests.get(url_details, params={
                "place_id": place_id,
                "fields": "formatted_phone_number,website",
                "key": API_KEY
            }).json()
            
            web = details.get("result", {}).get("website", "")
            telefono = details.get("result", {}).get("formatted_phone_number", "")
            email = extraer_email(web)
            whatsapp = extraer_whatsapp(web, telefono)
            
            nuevos_leads.append({
                "Nombre": place.get("name", ""),
                "Direccion": place.get("formatted_address", ""),
                "Telefono": telefono,
                "Web": web,
                "Rating": place.get("rating", ""),
                "Email": email,
                "Zona": zona,
                "place_id": place_id,
                "WhatsApp": whatsapp
            })
            print(f"OK: {place.get('name')}")
        
        next_page_token = response.get("next_page_token")
        if next_page_token:
            time.sleep(3)
            params = {"pagetoken": next_page_token, "key": API_KEY}
        else:
            break

df_nuevos = pd.DataFrame(nuevos_leads)
df_final = pd.concat([df_existente, df_nuevos], ignore_index=True)
df_final.to_excel("leads_odontologos_medellin_completo.xlsx", index=False)
print(f"\nNuevos leads agregados: {len(nuevos_leads)}")
print(f"Total base de datos: {len(df_final)}")
df_final


Buscando: centros odontologicos Copacabana Antioquia
OK: Consultorio Odontológico MCC ODONTOLOGY.
OK: Consultorio Odontológico MyDentale
OK: OralSalud Odontología Integral
OK: SonriDental
OK: Consultorio Odontológico Dra Natalia Zapata
OK: Consultorio Odontologico Lina Uribe
OK: D3Dental
Duplicado: Dental line
OK: Odontonorte
OK: Bocca Odontología Integral
OK: Centro odontológico siqueira campos
OK: Odontimax Odontologia
OK: Centro de Odontología Contemporánea

Buscando: centros odontologicos La Estrella Antioquia
OK: CLÍNICA DR. BLANC LA ESTRELLA
OK: o mas dental
OK: Dentart
OK: Dental Star Odontología
OK: Dontológica
OK: Solo Sonrisas - Odontología - Diseño Sonrisa- Estética facial
OK: Macridental
OK: centro dental bioser
Duplicado: Dra. Lisbed Giraldo Olarte, Odontólogo
OK: Consultorio Odontológico y psicologico Dra Natalia Tobon
OK: Consultorio Odontológico Adriana Aguirre
Duplicado: CREADENTISS, odontología integral
OK: Centro Odontológico Kr60

Buscando: centros odontologicos Ca

,Nombre,Direccion,Telefono,Web,Rating,Email,Zona,place_id,WhatsApp
0,Oral Center Poblado,"Calle 7 # 39- 197, Torre Intermédica Local 121...",(604) 3493080,http://oralcenter.com.co/,4.4,NaN,centros odontologicos El Poblado Medellin,ChIJX2o-xYGCRo4Rn6L7wacsRJo,NaN
1,Dental Center,"Cl. 11 #21, El Poblado, Medellín, El Poblado, ...",(604) 2686922,http://www.dentalcenter.com.co/,4.9,Pacientesdc@gmail.com,centros odontologicos El Poblado Medellin,ChIJgwmJfiYoRI4RvO8goH3oe7M,NaN
2,Centro odontológico KMO Smile Especialistas,"Av. El Poblado #cl 7 39-290, El Poblado, Medel...",314 6628875,NaN,4.9,NaN,centros odontologicos El Poblado Medellin,ChIJc8AO2KUpRI4R5lZzeBklf3U,573146628875.0
3,Poblado Dental,"Cl 10 #42-45 Oficina 229, El Poblado, Medellín...",324 6027040,https://pobladodental.com/,4.6,pobladodental@gmail.com,centros odontologicos El Poblado Medellin,ChIJ42xK_4kpRI4RwY-W14dE_BU,573246027040.0
4,Odontología el Poblado - Oral 10 - Diseños de ...,"Cra. 48 #7 270, El Poblado, Medellín, El Pobla...",301 1795872,http://www.oral10odontologia.com/,5.0,oral10vegas@gmail.com,centros odontologicos El Poblado Medellin,ChIJN3fIPZ8pRI4RytlAMwH5Zuw,573011795872.0
...,...,...,...,...,...,...,...,...,...
512,ORTHO ESTETIKA . SANJAVIER,"Cl 42 #101-73, Medellín, San Javier, Medellín,...",,,3.8,,centros odontologicos San Javier Medellin,ChIJXeYnq9opRI4R_rGC_3MDNao,
513,Koodent,"Cra. 86B #47A-27, La América, Medellín, La Amé...",311 3081680,https://www.instagram.com/koodent/,4.9,,centros odontologicos Popular Medellin,ChIJ3VwBe2spRI4R4U3AYHckB3I,+573113081680
514,CENTRO DENTAL POPULAR #2,"Cl. 55 #56A-64, La Candelaria, Medellín, La Ca...",45128881,,,,centros odontologicos Popular Medellin,ChIJoYCxof0oRI4RMOb-PJCxTBM,
515,Dental Force Santa Cruz,"Cra. 43a #104-26, Villa Del Socorro, Medellín,...",314 6940090,,5,,centros odontologicos Santa Cruz Medellin,ChIJD3e3GKsvRI4RP60CDYO6lBw,+573146940090


In [36]:
import pandas as pd

df = pd.read_excel("leads_odontologos_medellin_completo.xlsx")
print(df["Zona"].value_counts())

Zona
centros odontologicos Laureles Medellin        59
centros odontologicos Envigado                 58
centros odontologicos Itagui                   52
centros odontologicos El Poblado Medellin      51
centros odontologicos Bello                    46
centros odontologicos Centro Medellin          33
centros odontologicos Estadio Medellin         28
centros odontologicos Guayabal Medellin        27
centros odontologicos Sabaneta                 25
centros odontologicos Belen Medellin           22
centros odontologicos Castilla Medellin        19
centros odontologicos Manrique Medellin        18
centros odontologicos Copacabana Antioquia     12
centros odontologicos Caldas Antioquia         12
centros odontologicos La Estrella Antioquia    11
centros odontologicos Robledo Medellin         10
centros odontologicos Buenos Aires Medellin    10
centros odontologicos Aranjuez Medellin         9
centros odontologicos San Javier Medellin       7
centros odontologicos Floresta Medellin      

In [38]:
import pandas as pd

df = pd.read_excel("leads_odontologos_medellin_completo.xlsx")

# Mapeo de zonas a comunas
comunas = {
    "centros odontologicos Laureles Medellin": "Comuna 11 - Laureles Estadio",
    "centros odontologicos Estadio Medellin": "Comuna 11 - Laureles Estadio",
    "centros odontologicos Floresta Medellin": "Comuna 11 - Laureles Estadio",
    "centros odontologicos El Poblado Medellin": "Comuna 14 - El Poblado",
    "centros odontologicos Centro Medellin": "Comuna 10 - La Candelaria",
    "centros odontologicos Belen Medellin": "Comuna 16 - Belen",
    "centros odontologicos Robledo Medellin": "Comuna 7 - Robledo",
    "centros odontologicos Castilla Medellin": "Comuna 5 - Castilla",
    "centros odontologicos Guayabal Medellin": "Comuna 15 - Guayabal",
    "centros odontologicos Buenos Aires Medellin": "Comuna 9 - Buenos Aires",
    "centros odontologicos Manrique Medellin": "Comuna 3 - Manrique",
    "centros odontologicos Aranjuez Medellin": "Comuna 4 - Aranjuez",
    "centros odontologicos San Javier Medellin": "Comuna 13 - San Javier",
    "centros odontologicos Popular Medellin": "Comuna 1 - Popular",
    "centros odontologicos Santa Cruz Medellin": "Comuna 2 - Santa Cruz",
    "centros odontologicos Envigado": "Municipio - Envigado",
    "centros odontologicos Itagui": "Municipio - Itagui",
    "centros odontologicos Bello": "Municipio - Bello",
    "centros odontologicos Sabaneta": "Municipio - Sabaneta",
    "centros odontologicos Copacabana Antioquia": "Municipio - Copacabana",
    "centros odontologicos La Estrella Antioquia": "Municipio - La Estrella",
    "centros odontologicos Caldas Antioquia": "Municipio - Caldas"
}

df["Comuna"] = df["Zona"].map(comunas)

df.to_excel("leads_odontologos_medellin_completo.xlsx", index=False)
print(df["Comuna"].value_counts())
df

Comuna
Comuna 11 - Laureles Estadio    91
Municipio - Envigado            58
Municipio - Itagui              52
Comuna 14 - El Poblado          51
Municipio - Bello               46
Comuna 10 - La Candelaria       33
Comuna 15 - Guayabal            27
Municipio - Sabaneta            25
Comuna 16 - Belen               22
Comuna 5 - Castilla             19
Comuna 3 - Manrique             18
Municipio - Copacabana          12
Municipio - Caldas              12
Municipio - La Estrella         11
Comuna 7 - Robledo              10
Comuna 9 - Buenos Aires         10
Comuna 4 - Aranjuez              9
Comuna 13 - San Javier           7
Comuna 1 - Popular               2
Comuna 2 - Santa Cruz            2
Name: count, dtype: int64


,Nombre,Direccion,Telefono,Web,Rating,Email,Zona,place_id,WhatsApp,Comuna
0,Oral Center Poblado,"Calle 7 # 39- 197, Torre Intermédica Local 121...",(604) 3493080,http://oralcenter.com.co/,4.4,NaN,centros odontologicos El Poblado Medellin,ChIJX2o-xYGCRo4Rn6L7wacsRJo,NaN,Comuna 14 - El Poblado
1,Dental Center,"Cl. 11 #21, El Poblado, Medellín, El Poblado, ...",(604) 2686922,http://www.dentalcenter.com.co/,4.9,Pacientesdc@gmail.com,centros odontologicos El Poblado Medellin,ChIJgwmJfiYoRI4RvO8goH3oe7M,NaN,Comuna 14 - El Poblado
2,Centro odontológico KMO Smile Especialistas,"Av. El Poblado #cl 7 39-290, El Poblado, Medel...",314 6628875,NaN,4.9,NaN,centros odontologicos El Poblado Medellin,ChIJc8AO2KUpRI4R5lZzeBklf3U,5.731466e+11,Comuna 14 - El Poblado
3,Poblado Dental,"Cl 10 #42-45 Oficina 229, El Poblado, Medellín...",324 6027040,https://pobladodental.com/,4.6,pobladodental@gmail.com,centros odontologicos El Poblado Medellin,ChIJ42xK_4kpRI4RwY-W14dE_BU,5.732460e+11,Comuna 14 - El Poblado
4,Odontología el Poblado - Oral 10 - Diseños de ...,"Cra. 48 #7 270, El Poblado, Medellín, El Pobla...",301 1795872,http://www.oral10odontologia.com/,5.0,oral10vegas@gmail.com,centros odontologicos El Poblado Medellin,ChIJN3fIPZ8pRI4RytlAMwH5Zuw,5.730118e+11,Comuna 14 - El Poblado
...,...,...,...,...,...,...,...,...,...,...
512,ORTHO ESTETIKA . SANJAVIER,"Cl 42 #101-73, Medellín, San Javier, Medellín,...",NaN,NaN,3.8,NaN,centros odontologicos San Javier Medellin,ChIJXeYnq9opRI4R_rGC_3MDNao,NaN,Comuna 13 - San Javier
513,Koodent,"Cra. 86B #47A-27, La América, Medellín, La Amé...",311 3081680,https://www.instagram.com/koodent/,4.9,NaN,centros odontologicos Popular Medellin,ChIJ3VwBe2spRI4R4U3AYHckB3I,5.731131e+11,Comuna 1 - Popular
514,CENTRO DENTAL POPULAR #2,"Cl. 55 #56A-64, La Candelaria, Medellín, La Ca...",45128881,NaN,NaN,NaN,centros odontologicos Popular Medellin,ChIJoYCxof0oRI4RMOb-PJCxTBM,NaN,Comuna 1 - Popular
515,Dental Force Santa Cruz,"Cra. 43a #104-26, Villa Del Socorro, Medellín,...",314 6940090,NaN,5.0,NaN,centros odontologicos Santa Cruz Medellin,ChIJD3e3GKsvRI4RP60CDYO6lBw,5.731469e+11,Comuna 2 - Santa Cruz


In [39]:
import pandas as pd

df = pd.read_excel("leads_odontologos_medellin_completo.xlsx")
df["Categoria"] = "Salud"
df["Subcategoria"] = "Odontologia"
df.to_excel("leads_medellin_completo.xlsx", index=False)
print(f"Listo: {len(df)} registros actualizados")
print(df.columns.tolist())

Listo: 517 registros actualizados
['Nombre', 'Direccion', 'Telefono', 'Web', 'Rating', 'Email', 'Zona', 'place_id', 'WhatsApp', 'Comuna', 'Categoria', 'Subcategoria']


In [40]:
import requests
import pandas as pd
import time
import re

API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"
url_search = "https://maps.googleapis.com/maps/api/place/textsearch/json"
url_details = "https://maps.googleapis.com/maps/api/place/details/json"

# Cargar base existente
df_existente = pd.read_excel("leads_medellin_completo.xlsx")
ids_vistos = set(df_existente["place_id"].dropna().tolist())

zonas = [
    "clinicas estetica El Poblado Medellin",
    "clinicas estetica Laureles Medellin",
    "clinicas estetica Envigado",
    "clinicas estetica Bello",
    "clinicas estetica Itagui",
    "clinicas estetica Centro Medellin",
    "clinicas estetica Sabaneta",
    "clinicas estetica Belen Medellin",
    "clinicas estetica Robledo Medellin",
    "clinicas estetica Castilla Medellin",
    "clinicas estetica Buenos Aires Medellin",
    "clinicas estetica Manrique Medellin",
    "clinicas estetica Aranjuez Medellin",
    "clinicas estetica Copacabana Antioquia",
    "clinicas estetica La Estrella Antioquia",
    "clinicas estetica Caldas Antioquia",
    "clinicas estetica Guayabal Medellin",
    "clinicas estetica Estadio Medellin",
    "clinicas estetica Floresta Medellin",
    "clinicas estetica San Javier Medellin",
    "clinicas estetica Popular Medellin",
    "clinicas estetica Santa Cruz Medellin"
]

comunas = {
    "clinicas estetica Laureles Medellin": "Comuna 11 - Laureles Estadio",
    "clinicas estetica Estadio Medellin": "Comuna 11 - Laureles Estadio",
    "clinicas estetica Floresta Medellin": "Comuna 11 - Laureles Estadio",
    "clinicas estetica El Poblado Medellin": "Comuna 14 - El Poblado",
    "clinicas estetica Centro Medellin": "Comuna 10 - La Candelaria",
    "clinicas estetica Belen Medellin": "Comuna 16 - Belen",
    "clinicas estetica Robledo Medellin": "Comuna 7 - Robledo",
    "clinicas estetica Castilla Medellin": "Comuna 5 - Castilla",
    "clinicas estetica Guayabal Medellin": "Comuna 15 - Guayabal",
    "clinicas estetica Buenos Aires Medellin": "Comuna 9 - Buenos Aires",
    "clinicas estetica Manrique Medellin": "Comuna 3 - Manrique",
    "clinicas estetica Aranjuez Medellin": "Comuna 4 - Aranjuez",
    "clinicas estetica San Javier Medellin": "Comuna 13 - San Javier",
    "clinicas estetica Popular Medellin": "Comuna 1 - Popular",
    "clinicas estetica Santa Cruz Medellin": "Comuna 2 - Santa Cruz",
    "clinicas estetica Envigado": "Municipio - Envigado",
    "clinicas estetica Itagui": "Municipio - Itagui",
    "clinicas estetica Bello": "Municipio - Bello",
    "clinicas estetica Sabaneta": "Municipio - Sabaneta",
    "clinicas estetica Copacabana Antioquia": "Municipio - Copacabana",
    "clinicas estetica La Estrella Antioquia": "Municipio - La Estrella",
    "clinicas estetica Caldas Antioquia": "Municipio - Caldas"
}

nuevos_leads = []

def extraer_email(url):
    if not url or pd.isna(url):
        return ""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=8)
        emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', response.text)
        emails = [e for e in emails if not any(x in e for x in ['sentry', 'wix', 'google', 'schema', 'example'])]
        return emails[0] if emails else ""
    except:
        return ""

def extraer_whatsapp(web, telefono):
    if web and not pd.isna(web):
        wa = re.findall(r'wa\.me/(\d+)', str(web))
        if wa:
            return "+" + wa[0]
    if telefono and not pd.isna(telefono):
        numero = re.sub(r'\D', '', str(telefono))
        if numero.startswith('3') and len(numero) == 10:
            return "+57" + numero
        if numero.startswith('573') and len(numero) == 12:
            return "+" + numero
    return ""

for zona in zonas:
    print(f"\nBuscando: {zona}")
    params = {"query": zona, "key": API_KEY, "language": "es"}
    
    while True:
        response = requests.get(url_search, params=params).json()
        
        for place in response.get("results", []):
            place_id = place.get("place_id", "")
            if place_id in ids_vistos:
                print(f"Duplicado: {place.get('name')}")
                continue
            ids_vistos.add(place_id)
            
            details = requests.get(url_details, params={
                "place_id": place_id,
                "fields": "formatted_phone_number,website",
                "key": API_KEY
            }).json()
            
            web = details.get("result", {}).get("website", "")
            telefono = details.get("result", {}).get("formatted_phone_number", "")
            email = extraer_email(web)
            whatsapp = extraer_whatsapp(web, telefono)
            
            nuevos_leads.append({
                "Nombre": place.get("name", ""),
                "Direccion": place.get("formatted_address", ""),
                "Telefono": telefono,
                "Web": web,
                "Rating": place.get("rating", ""),
                "Email": email,
                "Zona": zona,
                "place_id": place_id,
                "WhatsApp": whatsapp,
                "Comuna": comunas.get(zona, ""),
                "Categoria": "Estetica",
                "Subcategoria": "Clinica Estetica"
            })
            print(f"OK: {place.get('name')}")
        
        next_page_token = response.get("next_page_token")
        if next_page_token:
            time.sleep(3)
            params = {"pagetoken": next_page_token, "key": API_KEY}
        else:
            break

df_nuevos = pd.DataFrame(nuevos_leads)
df_final = pd.concat([df_existente, df_nuevos], ignore_index=True)
df_final.to_excel("leads_medellin_completo.xlsx", index=False)
print(f"\nNuevos leads estetica: {len(nuevos_leads)}")
print(f"Total base de datos: {len(df_final)}")
df_final


Buscando: clinicas estetica El Poblado Medellin
OK: Wellnes Spa by Clínica Antienvejecimiento, spa Medellín, spa parejas, spa poblado, clínica estética, revitalización facial
OK: Dra.Pastrana Medicina Estética Medellín
OK: Massai Clínica - Clínica Estética en Medellín
OK: Clínica Especialistas del Poblado
OK: EVOLVE Clínica Estética
OK: CentroSthetic
OK: Botox and Fillers Medellin Dra Catalina Henao Medicina antienvejecimiento
OK: Brassia Estética Poblado
OK: Bonanova Medicina Estética y Bienestar
OK: Clínica Quantum
OK: Centro Médico Femo Estética - Medellín Poblado
OK: Dra Monica Pineda | MP Medical Center
OK: Naturalness | Medicina Estética
OK: Spa Médica - Dr. Juan Mario Escobar
OK: Integral Esthetic Botox / Facial / Fillers / Microagujas
OK: Siempreviva Medicina Estética Medellín Poblado
OK: Cirugía Plástica Medellin /Colombia Plastic Esthetic International
OK: Dr. Escorcia
OK: Alviva Clínica Estética
OK: Clínica Mery Álvarez - Sede Poblado
OK: Cirugia Plastica Medellin
OK: Medic

,Nombre,Direccion,Telefono,Web,Rating,Email,Zona,place_id,WhatsApp,Comuna,Categoria,Subcategoria
0,Oral Center Poblado,"Calle 7 # 39- 197, Torre Intermédica Local 121...",(604) 3493080,http://oralcenter.com.co/,4.4,NaN,centros odontologicos El Poblado Medellin,ChIJX2o-xYGCRo4Rn6L7wacsRJo,NaN,Comuna 14 - El Poblado,Salud,Odontologia
1,Dental Center,"Cl. 11 #21, El Poblado, Medellín, El Poblado, ...",(604) 2686922,http://www.dentalcenter.com.co/,4.9,Pacientesdc@gmail.com,centros odontologicos El Poblado Medellin,ChIJgwmJfiYoRI4RvO8goH3oe7M,NaN,Comuna 14 - El Poblado,Salud,Odontologia
2,Centro odontológico KMO Smile Especialistas,"Av. El Poblado #cl 7 39-290, El Poblado, Medel...",314 6628875,NaN,4.9,NaN,centros odontologicos El Poblado Medellin,ChIJc8AO2KUpRI4R5lZzeBklf3U,573146628875.0,Comuna 14 - El Poblado,Salud,Odontologia
3,Poblado Dental,"Cl 10 #42-45 Oficina 229, El Poblado, Medellín...",324 6027040,https://pobladodental.com/,4.6,pobladodental@gmail.com,centros odontologicos El Poblado Medellin,ChIJ42xK_4kpRI4RwY-W14dE_BU,573246027040.0,Comuna 14 - El Poblado,Salud,Odontologia
4,Odontología el Poblado - Oral 10 - Diseños de ...,"Cra. 48 #7 270, El Poblado, Medellín, El Pobla...",301 1795872,http://www.oral10odontologia.com/,5.0,oral10vegas@gmail.com,centros odontologicos El Poblado Medellin,ChIJN3fIPZ8pRI4RytlAMwH5Zuw,573011795872.0,Comuna 14 - El Poblado,Salud,Odontologia
...,...,...,...,...,...,...,...,...,...,...,...,...
1403,Carolina Nail's Spa,"Cr 52 cl 97#125 50003, Medellín, Antioquia, Co...",320 5260389,,,,clinicas estetica Santa Cruz Medellin,ChIJk7AeNtMvRI4RVB4O4g5GkvM,+573205260389,Comuna 2 - Santa Cruz,Estetica,Clinica Estetica
1404,steps,"Cl. 99 #49c47, Santa Cruz, Medellín, Santa Cru...",312 2385215,,,,clinicas estetica Santa Cruz Medellin,ChIJs1R3nd0vRI4R6tp8QdrWof8,+573122385215,Comuna 2 - Santa Cruz,Estetica,Clinica Estetica
1405,Whithe natural odontología integral,"Cl. 98 #51A - 82, La Rosa, Medellín, Santa Cru...",315 0001021,,5,,clinicas estetica Santa Cruz Medellin,ChIJn-HJEHgvRI4R9RIwbudRV2w,+573150001021,Comuna 2 - Santa Cruz,Estetica,Clinica Estetica
1406,Lux Nails,"Cl. 95 #52 -38, Aranjuez, Medellín, Aranjuez, ...",321 8848306,https://linktr.ee/luxnailsmde,,,clinicas estetica Santa Cruz Medellin,ChIJz8PlAAApRI4RoH_5TdZm4-w,+573218848306,Comuna 2 - Santa Cruz,Estetica,Clinica Estetica


In [ ]:
import requests
import pandas as pd
import re

df = pd.read_excel("leads_medellin_completo.xlsx")

# Filtrar solo estetica sin email
sin_email = df[(df["Subcategoria"] == "Clinica Estetica") & (df["Email"].isna() | (df["Email"] == ""))]
print(f"Registros de estetica sin email: {len(sin_email)}")

def extraer_email_profundo(url):
    if not url or pd.isna(url):
        return ""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=8)
        emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', response.text)
        emails = [e for e in emails if not any(x in e for x in ['sentry', 'wix', 'google', 'schema', 'example', 'jquery'])]
        if emails:
            return emails[0]
        
        from urllib.parse import urljoin
        paginas = [
            urljoin(url, "/contacto"),
            urljoin(url, "/contact"),
            urljoin(url, "/nosotros"),
            urljoin(url, "/about"),
            urljoin(url, "/contactenos"),
        ]
        for pagina in paginas:
            try:
                r = requests.get(pagina, headers=headers, timeout=6)
                if r.status_code == 200:
                    emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', r.text)
                    emails = [e for e in emails if not any(x in e for x in ['sentry', 'wix', 'google', 'schema', 'example', 'jquery'])]
                    if emails:
                        return emails[0]
            except:
                continue
        return ""
    except:
        return ""

for i, row in sin_email.iterrows():
    email = extraer_email_profundo(row.get("Web"))
    if email:
        df.at[i, "Email"] = email
        print(f"OK: {row['Nombre']} --> {email}")
    else:
        print(f"sin email: {row['Nombre']}")

df.to_excel("leads_medellin_completo.xlsx", index=False)
total_estetica = df[df["Subcategoria"] == "Clinica Estetica"]
print(f"\nEmails estetica: {total_estetica['Email'].ne('').sum()} de {len(total_estetica)}")

Registros de estetica sin email: 787
sin email: Wellnes Spa by Clínica Antienvejecimiento, spa Medellín, spa parejas, spa poblado, clínica estética, revitalización facial
sin email: Massai Clínica - Clínica Estética en Medellín
sin email: Clínica Especialistas del Poblado
sin email: EVOLVE Clínica Estética
sin email: Brassia Estética Poblado
sin email: Bonanova Medicina Estética y Bienestar
sin email: Centro Médico Femo Estética - Medellín Poblado
sin email: Dra Monica Pineda | MP Medical Center
sin email: Spa Médica - Dr. Juan Mario Escobar
sin email: Siempreviva Medicina Estética Medellín Poblado
sin email: Alviva Clínica Estética
sin email: Dra. Yulieth Figueroa
sin email: Dra Amy Barreto - Blefaroplastia Abdominoplastia Liposucción Mastopexia Lifting facial Labioplastia Lipoma Cirugía Varices
sin email: Cirugía Estética Avanzada - Cirugia Plastica
sin email: Anay Clínica Estética
sin email: Satori IPS Medicina Estética
sin email: Dr. Yesid Cardenas
sin email: Dr. Milton Rincón Carr

In [ ]:
import pandas as pd

df = pd.read_excel("leads_medellin_completo.xlsx")
estetica = df[df["Subcategoria"] == "Clinica Estetica"]
print(f"Total estetica: {len(estetica)}")
print(f"Con email: {estetica['Email'].ne('').sum()}")
print(f"Con WhatsApp: {estetica['WhatsApp'].ne('').sum()}")
print(f"Con Web: {estetica['Web'].ne('').sum()}")

In [1]:
import pandas as pd

df = pd.read_excel("leads_medellin_completo.xlsx")
estetica = df[df["Subcategoria"] == "Clinica Estetica"]
print(f"Total estetica: {len(estetica)}")
print(f"Con email: {estetica['Email'].ne('').sum()}")
print(f"Con WhatsApp: {estetica['WhatsApp'].ne('').sum()}")
print(f"Con Web: {estetica['Web'].ne('').sum()}")

Total estetica: 891
Con email: 891
Con WhatsApp: 891
Con Web: 891


In [2]:
import pandas as pd

df = pd.read_excel("leads_medellin_completo.xlsx")
estetica = df[df["Subcategoria"] == "Clinica Estetica"]

print(f"Total estetica: {len(estetica)}")
print(f"Con email: {estetica['Email'].notna().sum()}")
print(f"Sin email: {estetica['Email'].isna().sum()}")
print(f"Email vacio: {(estetica['Email'] == '').sum()}")
print(f"Con WhatsApp: {estetica['WhatsApp'].notna().sum()}")
print(f"Sin WhatsApp: {estetica['WhatsApp'].isna().sum()}")

Total estetica: 891
Con email: 104
Sin email: 787
Email vacio: 0
Con WhatsApp: 722
Sin WhatsApp: 169


In [1]:
import pandas as pd
import re
import time
import random
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

# ── CONFIGURACIÓN ──────────────────────────────────────────────
ARCHIVO = "leads_medellin_completo.xlsx"
PAUSA_MIN = 3
PAUSA_MAX = 7

def crear_driver():
    options = Options()
    options.add_argument("--headless")               # sin ventana
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    options.add_argument("--log-level=3")            # silencia logs de Chrome
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.set_page_load_timeout(15)
    return driver

FILTROS_EMAIL = ['sentry', 'wix', 'google', 'schema', 'example', 'jquery',
                 'png', 'jpg', 'svg', 'css', 'js', '@2x', 'webpack']

def extraer_emails_de_html(html):
    emails = re.findall(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', html)
    return [e for e in emails if not any(x in e.lower() for x in FILTROS_EMAIL)]

def extraer_email_selenium(driver, url):
    if not url or pd.isna(url):
        return ""

    paginas = [
        url,
        urljoin(url, "/contacto"),
        urljoin(url, "/contact"),
        urljoin(url, "/nosotros"),
        urljoin(url, "/about"),
        urljoin(url, "/contactenos"),
    ]

    for pagina in paginas:
        try:
            driver.get(pagina)
            time.sleep(2)  # espera a que cargue el JS

            # Busca en el HTML renderizado
            html = driver.page_source
            emails = extraer_emails_de_html(html)
            if emails:
                return emails[0]

            # También busca en mailto: links (más confiable)
            mailto_links = driver.find_elements("css selector", "a[href^='mailto:']")
            for link in mailto_links:
                href = link.get_attribute("href")
                if href:
                    email = href.replace("mailto:", "").split("?")[0].strip()
                    if email and "@" in email:
                        return email

        except WebDriverException:
            continue
        except Exception:
            continue

    return ""

# ── MAIN ───────────────────────────────────────────────────────
df = pd.read_excel(ARCHIVO)

sin_email = df[
    (df["Subcategoria"] == "Clinica Estetica") &
    (df["Email"].isna() | (df["Email"] == ""))
]
print(f"Clínicas estética sin email: {len(sin_email)}")
print("Iniciando Selenium...\n")

driver = crear_driver()
encontrados = 0

try:
    for i, row in sin_email.iterrows():
        email = extraer_email_selenium(driver, row.get("Web"))

        if email:
            df.at[i, "Email"] = email
            encontrados += 1
            print(f"✅ {row['Nombre']} --> {email}")
        else:
            print(f"❌ sin email: {row['Nombre']}")

        # Guarda cada 10 registros por si se interrumpe
        if encontrados % 10 == 0 and encontrados > 0:
            df.to_excel(ARCHIVO, index=False)
            print(f"   💾 Guardado parcial ({encontrados} emails nuevos)")

        time.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX))

finally:
    driver.quit()
    df.to_excel(ARCHIVO, index=False)
    df.to_excel(ARCHIVO, index=False)

total_estetica = df[df["Subcategoria"] == "Clinica Estetica"]
print(f"Emails estética: {total_estetica['Email'].ne('').sum()} de {len(total_estetica)}")
print(f"Emails nuevos encontrados esta sesión: {encontrados}")

ModuleNotFoundError: No module named 'selenium'

In [ ]:
import pandas as pd
import re
import time
import random
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

ARCHIVO     = "leads_medellin_completo.xlsx"
ARCHIVO_OUT = "leads_medellin_actualizado.xlsx"  # guarda aquí, no toca el original
PAUSA_MIN   = 3
PAUSA_MAX   = 7

def crear_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    options.add_argument("--log-level=3")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.set_page_load_timeout(15)
    return driver

FILTROS_EMAIL = ['sentry', 'wix', 'google', 'schema', 'example', 'jquery',
                 'png', 'jpg', 'svg', 'css', 'js', '@2x', 'webpack']

def extraer_emails_de_html(html):
    emails = re.findall(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', html)
    return [e for e in emails if not any(x in e.lower() for x in FILTROS_EMAIL)]

def extraer_email_selenium(driver, url):
    if not url or pd.isna(url):
        return ""
    for slug in ["", "/contacto", "/contact", "/nosotros", "/about", "/contactenos"]:
        try:
            driver.get(urljoin(url, slug) if slug else url)
            time.sleep(2)
            emails = extraer_emails_de_html(driver.page_source)
            if emails:
                return emails[0]
            for link in driver.find_elements("css selector", "a[href^='mailto:']"):
                href = link.get_attribute("href")
                if href:
                    email = href.replace("mailto:", "").split("?")[0].strip()
                    if email and "@" in email:
                        return email
        except:
            continue
    return ""

df = pd.read_excel(ARCHIVO)

sin_email = df[
    (df["Subcategoria"] == "Clinica Estetica") &
    (df["Email"].isna() | (df["Email"] == ""))
]
print(f"Clínicas estética sin email: {len(sin_email)}")

driver = crear_driver()
encontrados = 0

try:
    for i, row in sin_email.iterrows():
        email = extraer_email_selenium(driver, row.get("Web"))
        if email:
            df.at[i, "Email"] = email
            encontrados += 1
            print(f"✅ {row['Nombre']} --> {email}")
        else:
            print(f"❌ {row['Nombre']}")

        if encontrados > 0 and encontrados % 10 == 0:
            df.to_excel(ARCHIVO_OUT, index=False)
            print(f"   💾 Guardado parcial: {encontrados} emails")

        time.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX))

finally:
    driver.quit()
    df.to_excel(ARCHIVO_OUT, index=False)
    print(f"\n💾 Guardado en: {ARCHIVO_OUT}")

total = df[df["Subcategoria"] == "Clinica Estetica"]
print(f"Emails estética: {total['Email'].notna().sum()} de {len(total)}")
print(f"Emails nuevos esta sesión: {encontrados}")

Clínicas estética sin email: 787
✅ Wellnes Spa by Clínica Antienvejecimiento, spa Medellín, spa parejas, spa poblado, clínica estética, revitalización facial --> servicioalcliente@clinica-antienvejecimiento.com
❌ Massai Clínica - Clínica Estética en Medellín
❌ Clínica Especialistas del Poblado
❌ EVOLVE Clínica Estética


In [7]:
import sys
print(sys.executable)


C:\ProgramData\anaconda3\python.exe


In [8]:
import subprocess
subprocess.run([r"C:\Users\LENOVO\AppData\Local\Programs\Python\Python314\python.exe", 
                "-m", "pip", "install", "selenium", "webdriver-manager"])

CompletedProcess(args=['C:\\Users\\LENOVO\\AppData\\Local\\Programs\\Python\\Python314\\python.exe', '-m', 'pip', 'install', 'selenium', 'webdriver-manager'], returncode=0)

In [9]:
import sys
print(sys.version)

3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]


In [13]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "selenium", "webdriver-manager"])

CompletedProcess(args=['C:\\ProgramData\\anaconda3\\python.exe', '-m', 'pip', 'install', 'selenium', 'webdriver-manager'], returncode=0)

In [14]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "selenium", "webdriver-manager"], capture_output=False)

CompletedProcess(args=['C:\\ProgramData\\anaconda3\\python.exe', '-m', 'pip', 'install', 'selenium', 'webdriver-manager'], returncode=0)

In [15]:
import selenium
print(selenium.__version__)

4.41.0


In [16]:
import pandas
import requests
import openpyxl
import numpy
import matplotlib
import scipy
import sklearn

paquetes = {
    "pandas": pandas.__version__,
    "requests": requests.__version__,
    "openpyxl": openpyxl.__version__,
    "numpy": numpy.__version__,
    "matplotlib": matplotlib.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
}

for p, v in paquetes.items():
    print(f"{p}: {v}")

pandas: 2.2.3
requests: 2.32.3
openpyxl: 3.1.5
numpy: 2.1.3
matplotlib: 3.10.0
scipy: 1.15.3
sklearn: 1.6.1


In [17]:
import pandas
import requests
import openpyxl
import numpy
import matplotlib
import scipy
import sklearn

paquetes = {
    "pandas": pandas.__version__,
    "requests": requests.__version__,
    "openpyxl": openpyxl.__version__,
    "numpy": numpy.__version__,
    "matplotlib": matplotlib.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
}

for p, v in paquetes.items():
    print(f"{p}: {v}")

pandas: 2.2.3
requests: 2.32.3
openpyxl: 3.1.5
numpy: 2.1.3
matplotlib: 3.10.0
scipy: 1.15.3
sklearn: 1.6.1


In [18]:
import pandas
import requests
import openpyxl
import numpy
import matplotlib
import scipy
import sklearn

paquetes = {
    "pandas": pandas.__version__,
    "requests": requests.__version__,
    "openpyxl": openpyxl.__version__,
    "numpy": numpy.__version__,
    "matplotlib": matplotlib.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
}

for p, v in paquetes.items():
    print(f"{p}: {v}")

pandas: 2.2.3
requests: 2.32.3
openpyxl: 3.1.5
numpy: 2.1.3
matplotlib: 3.10.0
scipy: 1.15.3
sklearn: 1.6.1
